# Running Light Curves

In [ ]:
# --- In your notebook ---
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import sys
import pandas as pd
from itertools import islice
from IPython.display import display
import re
#imports
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import sys
import os
from astropy.cosmology import Planck18 as cosmo
import ast
import george
from collections import Counter
import pickle




In [ ]:
%reload_ext autoreload

In [ ]:


# If slsn_gp_lc.py lives in SLSNe_Metric/py_files/, add that parent to PYTHONPATH
repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
sys.path.insert(0, str(repo_root / "py_files"))  # contains slsn_gp_lc.py

# Import the module by name (matches filename without .py)
slsn_metric = importlib.import_module("local_SLSNe_metric")
importlib.reload(slsn_metric)
print("[CONFIG] Using Cristina's local MacBook setup")

# Import the module by name (matches filename without .py)
shared_utils = importlib.import_module("shared_utils")
importlib.reload(shared_utils)
print("[CONFIG] Using Cristina's local MacBook setup")

# Handy aliases
CatalogInputs = slsn_metric.CatalogInputs
SLSN_LC       = slsn_metric.LC



In [ ]:


repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
supernovae_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae")
out_perevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")

for csv in sorted(out_perevent.glob("*.csv")):
    # skip summary/derived files
    if csv.name.startswith("_") or csv.name.endswith("_cenwave.csv"):
        continue

    name = csv.stem  # no need to strip suffix now
    cen_map = slsn_metric.per_filter_cenwave(supernovae_dir, name, rel_bin=0.02, verbose=True)

    # Optional: audit variability (one-liner)
    diag_mode = slsn_metric.audit_cenwave_mode(supernovae_dir, name, rel_bin=0.02,
                                               outlier_frac_thresh=0.25)
    
    summary,var_df = slsn_metric.audit_cenwave_variability(supernovae_dir, name)
    
    if not var_df.empty and (var_df["flag_scatter"].any() or var_df["flag_trend"].any()):
        print(f"[audit] {name}: potential Cenwave variability flagged:")
        print(var_df.loc[var_df["flag_scatter"] | var_df["flag_trend"],
                         ["filter","rel_scatter","slope_A_per_day","r2"]])
    
    if not diag_mode.empty and diag_mode["flag_outliers"].any():
        print(f"[audit] {name}: filters with many outliers:")
        print(diag_mode.loc[diag_mode["flag_outliers"],
                            ["source","filter","n_rows","mode_center_A","frac_outside"]])


    slsn_metric.attach_cenwave_to_perevent_csv(csv, cen_map)



In [ ]:
# ========================================
# CONFIGURATION: Paths and Parameters
# ========================================

base_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/Rubin_tests")
per_event_csv_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")
allparams_csv = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/allparameter.csv")

# ========================================
# SLSN-SPECIFIC SIMULATION PARAMETERS
# ========================================

# Rate: SLSNe are rare - ~1-10% of core-collapse SNe
# Core-collapse rate ~1e-4 Mpc^-3 yr^-1, so SLSN ~1e-6 to 1e-5
rate_density = 3e-8 # Mpc^-3 yr^-1 (conservative mid-range)

# Redshift: SLSNe discovered out to z~4, but most < z~2
# Your catalog spans z~0.02-1.6, so match that range
z_min, z_max = 0.02, 0.8

# Survey duration
t_start, t_end = 1, 3652  # 10 years

# Extinction: Yes, SLSNe are extragalactic
use_extinction = True

# K-correction: For SLSNe, spectral index approach is less valid
# They have blue UV continuum but also complex line features
# Better to use your GP-fitted SEDs if possible, but for now:
use_kcorrect = False  # Will use your sed_grid when you implement it
k_correct_type = None  # Not applicable if use_kcorrect=False
k_correct_arg = None

# Distance constraints: None needed - z determines distance
d_min, d_max = None, None
gal_lat_cut = None  # SLSNe are extragalactic, no Galactic plane cut

# Cadences to test
cadences = ['four_roll_v4.3.1_10yrs']
ignore_triples = True  # Standard for metrics

# Control flags
generate_new_templates = False  # Set True only if templates don't exist
generate_new_pop = True
make_debug_plots = True

# ========================================
# SLSN TEMPLATE PARAMETERS
# ========================================

# These match SLSN timescales:
# - Rise: ~20-40 days (faster than SNe Ia)
# - Peak duration: ~10-20 days
# - Decline: t_50(r) ~ 30-60 days
n_time_steps = 220
tpad_pre_peak = 5.0   # days before peak (SLSNe rise quickly)
tpad_post_peak = 160.0  # days after peak (need ~100d for decline rate)

print("SLSN Simulation Configuration:")
print(f"  Rate: {rate_density:.1e} Mpc^-3 yr^-1")
print(f"  Redshift: {z_min} - {z_max}")
print(f"  Survey: {t_end-t_start} days")
print(f"  Expected events (rough): ~{int(rate_density * 1e10 * (t_end-t_start)/365)}")

In [ ]:


# ========================================
# BUILD OUTPUT PATHS (includes templates_file)
# ========================================

templates_file_str, pop_file_str, df_file_str, storage_dir_str, summary_filename_str = shared_utils.build_filenames(
    rate_density=rate_density, 
    z_min=z_min, 
    z_max=z_max, 
    d_min=d_min, 
    d_max=d_max,
    science_case="SLSNe", 
    testname=None, 
    testname_metric_only=None,
    ignore_triples=ignore_triples, 
    use_extinction=use_extinction, 
    use_kcorrect=use_kcorrect,
    base_dir=base_dir
)

# Convert to Path objects
templates_file = Path(templates_file_str)
pop_file = Path(pop_file_str)
df_file = Path(df_file_str)
storage_dir = Path(storage_dir_str)
summary_filename = Path(summary_filename_str)

print(f"Output paths:")
print(f"  Templates: {templates_file}")
print(f"  Population: {pop_file}")
print(f"  Results: {df_file}")

# ========================================
# BUILD TEMPLATES
# ========================================

if not templates_file.exists():
    print(f"\n[BUILD] Creating templates...")
    
    # Ensure parent directory exists
    templates_file.parent.mkdir(parents=True, exist_ok=True)
    
    # Build from cenwave-enriched CSVs
    inputs = CatalogInputs(
        photometry_dir=per_event_csv_dir,
        params_table=pd.read_csv(allparams_csv),
        name_col="name",
        z_col="redshift_med",
        peak_mjd_col="Peak_MJD_med"
    )
    
    print(f"Processing {len(inputs.params_table)} events...")
    
    # This fits 2D GPs - will take several minutes
    shared_lc_model = SLSN_LC.from_catalog(
        inputs,
        filename_pattern="{name}_cenwave.csv",
        save_to=templates_file,
        n_time=220,
        tpad_pre_days=5.0,
        tpad_post_days=160.0,
        min_points_for_fit=6,
    )
    
    print(f"✓ Built {len(shared_lc_model.data)} templates")
    
else:
    print(f"\n[LOAD] Templates already exist at {templates_file}")
    shared_lc_model = SLSN_LC(load_from=templates_file)
    print(f"✓ Loaded {len(shared_lc_model.data)} templates")

shared_lc_model.build_magnitude_grid(
    z_grid=np.linspace(0.02, 2.0, 50),
    phase_grid=np.geomspace(0.1, 160, 200),
    filters='ugrizy',
    save_to=grid_file
)

grid_file = templates_file.parent / f"{templates_file.stem}_mag_grid.pkl"


#print(f"✓ Bands: {shared_lc_model.filts}")
print(f"✓ First 5 events: {shared_lc_model.names[:5]}")



# Ignore these

In [ ]:


event = "2011kf"

# find the template index for this event
# find the correct (z, t0)
idx = slsn_metric.template_index_for_event(templates_file, "2011kf")
z, t0 = slsn_metric.get_event_z_t0("2011kf", inputs.params_table, per_event_csv_dir)

# 1) observations (name, not index)
slsn_metric.plot_event_obs("2011kf",
    per_event_dir=per_event_csv_dir, use_phase=True, z=z, t0=t0, scale_to_peak=False)


# 2) model curves (index)
slsn_metric.plot_event_model(
    templates_file,
    template_idx=idx
)

# 3) overlay obs + model (name + index)
slsn_metric.plot_event_obs_vs_model(
    "2011kf",
    per_event_dir=per_event_csv_dir,
    templates_file=templates_file,
    template_idx=idx,
    use_phase=True, z=z, t0=t0,
    bands=["g","r","i","z"],
    scale_to_peak=False,
)






# Diagnostics

In [ ]:


# 1) What filters exist in 2011kf_cenwave.csv?
p = per_event_csv_dir / "2011kf_cenwave.csv"
df = pd.read_csv(p)
cols = {c.lower(): c for c in df.columns}
print(sorted(df[cols["filter"]].astype(str).unique()))

# 2) For a given template index, list the model bands it actually has:
obj = pickle.load(open(templates_file, "rb"))
tpl = obj["lightcurves"][idx]   # use the idx you computed for 2011kf
print(sorted([k for k,v in tpl.items() if isinstance(v, dict) and {"ph","mag"}<=set(v)]))

# 3) Which events have a 'C' band? 
names = obj.get("names", [f"tpl_{i}" for i in range(len(obj["lightcurves"]))])
has_C = [n for n, tpl in zip(names, obj["lightcurves"])
         if any(k == "C" and isinstance(v, dict) and {"ph","mag"} <= set(v)
                for k, v in tpl.items())]
print("Events with C:", has_C)



In [ ]:


# 1) Apparent vs (Abs + DM)
df_res = slsn_metric.diagnose_abs_from_templates(
    event, per_event_csv_dir, templates_file, idx,
    z=z, t0=t0, bands=["g","r","i","z"], return_with_phase=True, verbose=True
)
slsn_metric.plot_residuals(df_res, by_band=True, title=f"{event}: m_obs − (M+DM)")
slsn_metric.plot_residuals(df_res, vs_phase=True, title=f"{event}: Residual vs Phase")

# 2) Internal GP consistency
df_int = slsn_metric.diagnose_internal_consistency(
    event, per_event_csv_dir, templates_file, idx,
    z=z, t0=t0, n_time=220, verbose=True, refit_gp = False
)
slsn_metric.plot_residuals(df_int, by_band=True, title=f"{event}: GP − (M+DM)")
slsn_metric.plot_residuals(df_int, vs_phase=True, title=f"{event}: GP Residual vs Phase")

# Quick audits
print("CSV bands:", slsn_metric.list_event_bands(per_event_csv_dir, event))
print("Template bands:", slsn_metric.list_template_bands(templates_file, idx))

# Continue here

In [ ]:

# Now this will work:
df_cov = characterize_template_coverage(templates_file)
print(f"\n{len(df_cov)} templates loaded")
print(f"Bands per template: {df_cov['n_bands'].describe()}")
print(f"Phase coverage: {df_cov['phase_span'].describe()}")
print(f"Most common bands: {df_cov['bands'].value_counts().head()}")

In [ ]:
def compute_slsn_properties(df_cov, templates_file):
    """
    Compute SLSN-specific observable properties from templates.
    """
    import pickle
    import numpy as np
    
    with open(templates_file, 'rb') as f:
        obj = pickle.load(f)
    
    lcs = obj['lightcurves']
    
    rows = []
    for i, tpl in enumerate(lcs):
        row = {'tpl_idx': i}
        
        # Get r-band curve
        if 'r' in tpl and isinstance(tpl['r'], dict):
            ph = np.asarray(tpl['r']['ph'], float)
            mag = np.asarray(tpl['r']['mag'], float)
            
            if len(ph) > 5:
                # Peak absolute magnitude
                peak_idx = np.argmin(mag)
                row['M_peak_r'] = float(mag[peak_idx])
                
                # Decline rate: 15-50 days post-peak
                post = (ph > 0) & (ph >= 15) & (ph <= 50)
                if post.sum() >= 3:
                    from scipy.stats import linregress
                    slope, _, _, _, _ = linregress(ph[post], mag[post])
                    row['decline_rate_r'] = float(slope)
                
                # Width at M_peak + 1 mag
                thresh = row['M_peak_r'] + 1.0
                above = mag < thresh
                if above.sum() >= 2:
                    row['width_r'] = float(ph[above].max() - ph[above].min())
        
        # g-r color at peak
        if 'g' in tpl and 'r' in tpl:
            g_mag = np.asarray(tpl['g']['mag'], float)
            g_ph = np.asarray(tpl['g']['ph'], float)
            r_mag = np.asarray(tpl['r']['mag'], float)
            r_ph = np.asarray(tpl['r']['ph'], float)
            
            # Find mags closest to phase=0
            if len(g_ph) > 0 and len(r_ph) > 0:
                g_near_peak = g_mag[np.argmin(np.abs(g_ph))]
                r_near_peak = r_mag[np.argmin(np.abs(r_ph))]
                row['g_minus_r'] = float(g_near_peak - r_near_peak)
        
        rows.append(row)
    
    df_props = pd.DataFrame(rows)
    
    # Merge with coverage info
    return df_cov.merge(df_props, on='tpl_idx', how='left')

# Compute properties FIRST
df_cov_full = compute_slsn_properties(df_cov, templates_file)

# NOW assess coverage
gaps = slsn_metric.assess_literature_coverage(df_cov_full)

In [ ]:

missing = slsn_metric.find_missing_archetypes(df_cov)

# Generating Population, Measuring Detection Efficiency 

In [ ]:


#Generate population using SLSN-specific function
slicer = generate_SLSN_PopSlicer(
    lc_model=shared_lc_model,
    peak_t_min=60980.5 + 300,
    peak_t_max=60980.5 + 2500,
    z_min=0.02,
    z_max=0.8,  # ← Lower this
    rate_density=3e-8,
    seed=42,
    save_to=pop_file
)


print(f"✓ Generated/loaded {len(slicer.slice_points['z'])} SLSN events")
print(f"  z range: {slicer.slice_points['z'].min():.3f} - {slicer.slice_points['z'].max():.3f}")
print(f"  peak times: {slicer.slice_points['peak_time'].min():.1f} - {slicer.slice_points['peak_time'].max():.1f} days")

# Optional: Quick diagnostics
if make_debug_plots:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    
    # Redshift distribution
    axes[0].hist(slicer.slice_points['z'], bins=30, edgecolor='k', alpha=0.7)
    axes[0].set_xlabel('Redshift')
    axes[0].set_ylabel('N events')
    axes[0].set_title('Redshift Distribution')
    
    # Peak time distribution
    axes[1].hist(slicer.slice_points['peak_time'], bins=30, edgecolor='k', alpha=0.7)
    axes[1].set_xlabel('Peak Time (days)')
    axes[1].set_ylabel('N events')
    axes[1].set_title('Peak Time Distribution')
    
    # Sky distribution
    axes[2].scatter(slicer.slice_points['ra'], slicer.slice_points['dec'], 
                   s=1, alpha=0.5)
    axes[2].set_xlabel('RA (deg)')
    axes[2].set_ylabel('Dec (deg)')
    axes[2].set_title('Sky Distribution')
    axes[2].set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

In [ ]:


slicer = generate_SLSN_PopSlicer(
    lc_model=shared_lc_model,
    peak_t_min=60980.5 + 300,
    peak_t_max=60980.5 + 2500,
    z_min=0.02,
    z_max=0.8,  # ← Lower this
    rate_density=3e-8,
    seed=42,
    save_to=pop_file
)

print(f"✓ Generated/loaded {len(slicer.slice_points['z'])} SLSN events")

# Then pass the SLICER OBJECT (not the function) to run_detect
from shared_utils import run_detect

df_obs = run_detect(
    metric=slsn_metric,           # your metric MODULE
    slicer=slicer,                # ← PASS THE SLICER OBJECT, NOT THE FUNCTION
    cadences=['four_roll_v4.3.1_10yrs'],
    shared_lc_model=shared_lc_model,  # ← PASS THE LC MODEL OBJECT, NOT LC.from_catalog
    db_dir=str(db_dir),           # path to your opsim databases
    storage_dir=str(storage_dir),
    df_file=str(df_file),
    use_extinction=True,
    use_kcorrect=False,
    is_grb=False,
    debug=True,
    plot=True
)

In [ ]:
# lean counting runs (no per-visit storage)
metric = SLSN_Detect_Metric(lc_model=lc, store_obs_mode="meta", diag_store=False)
1c) 
# keep only diagnostic subsamples (no full arrays)
metric = SLSN_Detect_Metric(lc_model=lc, store_obs_mode="diag")  # auto-enables diag_store

# keep full arrays (and diag subsamples if you set diag_store=True)
metric = SLSN_Detect_Metric(lc_model=lc, store_obs_mode="full", diag_store=False)  # full only
metric = SLSN_Detect_Metric(lc_model=lc, store_obs_mode="full", diag_store=True)   # full + diag

2a) 
# offline (once):
lc = LC(load_from=templates_pkl).build_magnitude_grid(
    z_grid=np.linspace(0.02, 2.0, 50),
    phase_grid=np.geomspace(0.1, 160.0, 200),
    filters='ugrizy',
    save_to=Path("slsn_maggrid.pkl")
)

# online (many runs):
lc = LC(load_from=templates_pkl).load_magnitude_grid("slsn_maggrid.pkl")
metric = SLSN_Detect_Metric(lc_model=lc, store_obs_mode="meta")
# run MAF; no grid or interp builds occur during the loop